## 1. Importar datos GATO GATO GATO

In [ ]:
import pandas as pd

df = pd.read_csv("data/kiva_loans.csv")
df.head()

,id,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,partner_id,posted_time,disbursed_time,funded_time,term_in_months,lender_count,tags,borrower_genders,repayment_interval,date
0,653051,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:12:39+00:00,2013-12-17 08:00:00+00:00,2014-01-02 10:06:32+00:00,12.0,12,NaN,female,irregular,2014-01-01
1,653053,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:51:08+00:00,2013-12-17 08:00:00+00:00,2014-01-02 09:17:23+00:00,11.0,14,NaN,"female, female",irregular,2014-01-01
2,653068,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,334.0,2014-01-01 09:58:07+00:00,2013-12-17 08:00:00+00:00,2014-01-01 16:01:36+00:00,43.0,6,"user_favorite, user_favorite",female,bullet,2014-01-01
3,653063,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 08:03:11+00:00,2013-12-24 08:00:00+00:00,2014-01-01 13:00:00+00:00,11.0,8,NaN,female,irregular,2014-01-01
4,653084,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,245.0,2014-01-01 11:53:19+00:00,2013-12-17 08:00:00+00:00,2014-01-01 19:18:51+00:00,14.0,16,NaN,female,monthly,2014-01-01


In [ ]:
df.shape

(671205, 20)

**Observación:** el dataset original contiene 671.205 registros (préstamos) y 20 columnas. Este número se usará como referencia para comparar contra el dataset final, después de la limpieza.

## 2. Análisis exploratorio rápido

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 671205 entries, 0 to 671204
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  671205 non-null  int64  
 1   funded_amount       671205 non-null  float64
 2   loan_amount         671205 non-null  float64
 3   activity            671205 non-null  str    
 4   sector              671205 non-null  str    
 5   use                 666973 non-null  str    
 6   country_code        671197 non-null  str    
 7   country             671205 non-null  str    
 8   region              614405 non-null  str    
 9   currency            671205 non-null  str    
 10  partner_id          657698 non-null  float64
 11  posted_time         671205 non-null  str    
 12  disbursed_time      668809 non-null  str    
 13  funded_time         622874 non-null  str    
 14  term_in_months      671205 non-null  float64
 15  lender_count        671205 non-null  int64  


**Observaciones:**
- Las columnas `posted_time`, `disbursed_time`, `funded_time` y `date` están tipadas como texto (`object`) en vez de fecha. Se convertirán a `datetime` en la sección de transformaciones.
- Hay valores faltantes en varias columnas, siendo `tags` la más afectada (~171.000 valores faltantes, ~25% del dataset). Se analizará cada caso en la sección de diagnóstico.

In [ ]:
df.describe()

,id,funded_amount,loan_amount,partner_id,term_in_months,lender_count
count,6.712050e+05,671205.000000,671205.000000,657698.000000,671205.000000,671205.000000
mean,9.932486e+05,785.995061,842.397107,178.199616,13.739022,20.590922
std,1.966113e+05,1130.398941,1198.660073,94.247581,8.598919,28.459551
min,6.530470e+05,0.000000,25.000000,9.000000,1.000000,0.000000
25%,8.230720e+05,250.000000,275.000000,126.000000,8.000000,7.000000
50%,9.927800e+05,450.000000,500.000000,145.000000,13.000000,13.000000
75%,1.163653e+06,900.000000,1000.000000,204.000000,14.000000,24.000000
max,1.340339e+06,100000.000000,100000.000000,536.000000,158.000000,2986.000000


**Observación:** en `loan_amount` y `funded_amount`, el percentil 75 no supera los $1.000, pero el valor máximo llega a $100.000 — una diferencia muy grande que sugiere posibles outliers. Se investigarán en detalle a continuación.

## 3. Diagnóstico de problemas

In [ ]:
df.sort_values(by="loan_amount", ascending=False).head(10)

,id,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,partner_id,posted_time,disbursed_time,funded_time,term_in_months,lender_count,tags,borrower_genders,repayment_interval,date
70499,722883,100000.0,100000.0,Agriculture,Agriculture,create more than 300 jobs for women and farmer...,HT,Haiti,Les Cayes,USD,315.0,2014-06-10 19:25:02+00:00,2014-09-08 07:00:00+00:00,2014-06-19 20:21:04+00:00,75.0,2986,"user_favorite, user_favorite, user_favorite, u...",female,irregular,2014-06-10
509048,1169175,50000.0,50000.0,Poultry,Agriculture,to purchase chicken feed & a delivery vehicle ...,TZ,Tanzania,Dar es Salaam,USD,497.0,2016-10-17 13:14:54+00:00,2016-12-31 08:00:00+00:00,2016-10-21 15:29:16+00:00,14.0,1765,"#Animals, #Woman Owned Biz, #Job Creator, #Biz...",female,irregular,2016-10-17
544548,1205071,50000.0,50000.0,Health,Health,to provide community trauma services in South ...,SS,South Sudan,Juba,USD,509.0,2016-12-16 17:07:25+00:00,2017-01-31 08:00:00+00:00,2016-12-18 07:04:32+00:00,8.0,1609,"#Female Education, #Health and Sanitation, #Un...",female,bullet,2016-12-16
583307,1245201,50000.0,50000.0,Agriculture,Agriculture,to support 800+ farmers by improving their pro...,GT,Guatemala,Quetzaltenango,USD,517.0,2017-02-23 23:49:12+00:00,2017-03-31 07:00:00+00:00,2017-03-22 23:32:13+00:00,20.0,1671,"user_favorite, user_favorite, user_favorite, u...",male,monthly,2017-02-23
492809,1152957,50000.0,50000.0,Agriculture,Agriculture,"to expand weather, farming information and fin...",GH,Ghana,Accra,USD,490.0,2016-09-19 13:08:02+00:00,2016-11-30 08:00:00+00:00,2016-10-17 18:36:56+00:00,14.0,1481,"#Technology, #Technology, #Sustainable Ag, use...",male,irregular,2016-09-19
126839,777718,50000.0,50000.0,Agriculture,Agriculture,to buy and plant resin producing pine trees. T...,MX,Mexico,Cherán,USD,376.0,2014-10-01 20:46:15+00:00,2014-08-31 07:00:00+00:00,2014-12-07 17:02:10+00:00,144.0,586,"user_favorite, user_favorite, #Biz Durable Ass...","male, male, male, male, male, male, male, female",irregular,2014-10-01
507237,1167661,9975.0,50000.0,Fruits & Vegetables,Food,to generate income opportunities for 1250+ low...,PE,Peru,Lima,USD,429.0,2016-10-13 19:32:42+00:00,2016-11-11 08:00:00+00:00,NaN,60.0,224,"user_favorite, user_favorite, user_favorite, u...",male,irregular,2016-10-13
490191,1150277,50000.0,50000.0,Health,Health,To purchase raw materials in order to produce ...,GH,Ghana,Accra,USD,489.0,2016-09-14 13:03:24+00:00,2016-11-30 08:00:00+00:00,2016-09-19 20:06:54+00:00,14.0,1569,"#Health and Sanitation, #Biz Durable Asset, #E...",male,irregular,2016-09-14
563074,1223392,50000.0,50000.0,Renewable Energy Products,Retail,to provide life-changing clean cookstoves and ...,KE,Kenya,Nairobi,USD,512.0,2017-01-20 00:49:43+00:00,2017-02-28 08:00:00+00:00,2017-01-28 17:04:15+00:00,14.0,1402,"#Eco-friendly, #Technology, user_favorite, use...",female,irregular,2017-01-20
43182,695450,50000.0,50000.0,Renewable Energy Products,Retail,To buy and sell Barefoot Power's solar lightin...,KE,Kenya,Nairobi,USD,212.0,2014-04-09 08:25:02+00:00,2014-06-09 07:00:00+00:00,2014-04-19 19:27:30+00:00,16.0,1491,"user_favorite, user_favorite, user_favorite, u...",male,bullet,2014-04-09


**Observación:** los préstamos más altos (entre $50.000 y $100.000) corresponden a proyectos comunitarios/agrícolas reales, con descripciones detalladas y coherentes (Haití, Tanzania, Guatemala, Sudán del Sur, etc.). No parecen errores de carga de datos, sino préstamos legítimos de mayor escala. **Decisión: se conservan.**

In [ ]:
df[df["funded_amount"] < df["loan_amount"]]["funded_time"].isnull().mean()

np.float64(1.0)

**Observación:** se confirma que el 100% de los préstamos con `funded_amount` menor a `loan_amount` (es decir, no financiados completamente) tienen `funded_time` vacío. Esto no es un error de carga: no existe una fecha de financiamiento completo porque el préstamo nunca alcanzó su meta. **Decisión:** estos valores nulos se dejarán como están (o se marcarán explícitamente), ya que representan información real ("préstamo aún no financiado"), no un dato faltante por error.

In [ ]:
df.duplicated().sum()

np.int64(0)

**Observación:** no se detectaron filas completamente duplicadas (`df.duplicated().sum()` = 0).

In [ ]:
df["id"].duplicated().sum()

np.int64(0)

**Observación:** la columna `id` no tiene valores duplicados (`df["id"].duplicated().sum()` = 0), confirmando que cada préstamo tiene un identificador único. Combinado con el chequeo anterior (sin filas completamente duplicadas), se concluye que este dataset no presenta problemas de duplicación.

In [ ]:
df.isnull().sum().sort_values(ascending=False)

tags                  171416
region                 56800
funded_time            48331
partner_id             13507
use                     4232
borrower_genders        4221
disbursed_time          2396
country_code               8
activity                   0
loan_amount                0
funded_amount              0
id                         0
posted_time                0
currency                   0
sector                     0
country                    0
lender_count               0
term_in_months             0
repayment_interval         0
date                       0
dtype: int64

**Observación — resumen de valores faltantes (de mayor a menor):**
- `tags`: 171.416 (~25.5% del dataset) — la más afectada
- `region`: 56.800 (~8.5%)
- `funded_time`: 48.331 (~7.2%) — ya explicado: corresponde a préstamos no financiados completamente
- `partner_id`: 13.507 (~2%)
- `use`: 4.232 (~0.6%)
- `borrower_genders`: 4.221 (~0.6%)
- `disbursed_time`: 2.396 (~0.4%)
- `country_code`: 8 (insignificante)

In [ ]:
df["tags"].head(10)

0                             NaN
1                             NaN
2    user_favorite, user_favorite
3                             NaN
4                             NaN
5                             NaN
6    user_favorite, user_favorite
7      #Elderly, #Woman Owned Biz
8                   user_favorite
9                             NaN
Name: tags, dtype: str

un pato